In [0]:
--SELECT version();
--USE DATABASE qdp_database;

CREATE SCHEMA IF NOT EXISTS qdp_transactions_all;
SET search_path TO qdp_transactions_all;


-- metadata table
DROP TABLE IF EXISTS metadata;


CREATE TABLE metadata ( 
	supplier VARCHAR,
	data_product VARCHAR,
	data_product_version VARCHAR,
	schema_name VARCHAR,
	schema_version VARCHAR,
	schema_variant_name VARCHAR,
	schema_variant_version VARCHAR,
	instance VARCHAR,
	instance_version VARCHAR
 );



INSERT INTO metadata 
(
  supplier, 
  data_product,
  data_product_version,
  schema_name,
  schema_version,
  schema_variant_name,
  schema_variant_version,
  instance,
  instance_version
) 
VALUES (
  'quantexa.com',
  'com.quantexa.qdp.transactions_all',
  '0.1',
  'transactions_all.data_vault',
  '0.1',
  'base',
  '0.1',
  'not yet populated with data',
  ''
);




-- -----------------------------------------------------------
-- create hub_transaction table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS hub_transaction CASCADE;

CREATE TABLE hub_transaction (
    transaction_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
  	tennant_id BIGINT NOT NULL DEFAULT 0,
    bene_account_entity_id VARCHAR,
    transaction_entity_id VARCHAR,
    orig_account_entity_id VARCHAR,
    bene_account_id BIGINT,
    orig_account_id BIGINT,
    transaction_id_raw VARCHAR,
		period_start TIMESTAMP,
		period_end TIMESTAMP
);

ALTER TABLE hub_transaction ADD CONSTRAINT pk_transaction_hubtransaction_transactionid PRIMARY KEY (transaction_id);

COMMENT ON COLUMN hub_transaction.transaction_id IS 'Transaction Identity';



-- -----------------------------------------------------------
-- create sat_transaction table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_transaction;
CREATE TABLE sat_transaction (
    transaction_id BIGINT NOT NULL,
    load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    record_source_id BIGINT NOT NULL DEFAULT 0,
	  type_code VARCHAR,
	  type_concept_id BIGINT,
	  type_raw_code VARCHAR,
	  type_raw_concept_id BIGINT,
	  data_source_code VARCHAR,
	  data_source_concept_id BIGINT,
	  data_source_raw_code VARCHAR,
	  data_source_raw_concept_id BIGINT,
	  debit_or_credit_code VARCHAR,
	  debit_or_credit_concept_id BIGINT,
	  debit_or_credit_raw_code VARCHAR,
	  debit_or_credit_raw_concept_id BIGINT,
		counterparty_account_id VARCHAR,
		counterparty_account_transaction_id VARCHAR,
		is_self_transaction BOOLEAN,
		is_international_transaction BOOLEAN,
    amount DECIMAL(38,18),
		datetime TIMESTAMP,
    description VARCHAR,
		balance_after DECIMAL(38,18),
		balance_before DECIMAL(38,18),	
	  payment_method_code VARCHAR,
	  payment_method_concept_id BIGINT,
	  payment_method_raw_code VARCHAR,
	  payment_method_raw_concept_id BIGINT,
    reference VARCHAR,
		account_sort_code VARCHAR,
		account_number VARCHAR,
		account_number_suffix VARCHAR,
    iban VARCHAR,
		swiftbic VARCHAR,
		beneficiary_name VARCHAR,
	  currency_code VARCHAR,
	  currency_concept_id BIGINT,
	  currency_raw_code VARCHAR,
	  currency_raw_concept_id BIGINT,
	  country_code VARCHAR,
	  country_concept_id BIGINT,
	  country_raw_code VARCHAR,
	  country_raw_concept_id BIGINT,
    original_amount DECIMAL(38,18),
		original_account_number VARCHAR,
		original_account_sort_code VARCHAR,
		original_iban VARCHAR,
		original_account_name VARCHAR,
		original_currency_code VARCHAR,
		original_currency_concept_id BIGINT,
	  original_currency_raw_code VARCHAR,
	  original_currency_raw_concept_id BIGINT,
		original_country_code VARCHAR,
		original_country_concept_id BIGINT,
	  original_country_raw_code VARCHAR,
	  original_country_raw_concept_id BIGINT,
		original_bank VARCHAR,
    original_bank_country_code VARCHAR,
	  institution_bank VARCHAR,
    institution_bank_country_code VARCHAR,
	  sending_bank VARCHAR,
    sending_bank_country_code VARCHAR,
    beneficiary_bank VARCHAR,
    beneficiary_bank_country_code VARCHAR,
    payment_booking_date DATE,
	  payment_booking_system_code VARCHAR,
	  payment_booking_system_concept_id BIGINT,
	  payment_booking_system_raw_code VARCHAR,
	  payment_booking_system_raw_concept_id BIGINT,
	  payment_type_code VARCHAR,
	  payment_type_concept_id BIGINT,
	  payment_type_raw_code VARCHAR,
	  payment_type_raw_concept_id BIGINT,
		other_receiver_information VARCHAR,
		payment_party_id BIGINT,
		payment_account_number VARCHAR,
		payment_source_code VARCHAR,
		period_start TIMESTAMP,
		period_end TIMESTAMP
);

ALTER TABLE sat_transaction ADD CONSTRAINT pk_transaction_sattransactions_transactionsid PRIMARY KEY (transaction_id);
ALTER TABLE sat_transaction ADD CONSTRAINT fk_transaction_sattransactions_hubtransaction_transactionid FOREIGN KEY (transaction_id) REFERENCES hub_transaction(transaction_id);

COMMENT ON COLUMN sat_transaction.transaction_id IS 'Transaction Identity';



-- -----------------------------------------------------------
-- create sat_transaction_party table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_transaction_party;

CREATE TABLE sat_transaction_party (
    transaction_party_id BIGINT NOT NULL,
    transaction_id BIGINT NOT NULL,
    load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    record_source_id BIGINT NOT NULL DEFAULT 0,
	  type_code VARCHAR,
	  type_concept_id BIGINT,
	  type_raw_code VARCHAR,
	  type_raw_concept_id BIGINT,
	  rank INT,
		period_start TIMESTAMP,
		period_end TIMESTAMP
);

ALTER TABLE sat_transaction_party ADD CONSTRAINT pk_transaction_sattransactionparty_transactionpartyid PRIMARY KEY (transaction_party_id);
ALTER TABLE sat_transaction_party ADD CONSTRAINT fk_transaction_sattransactionparty_hubtransactions_transactionid FOREIGN KEY (transaction_id) REFERENCES hub_transaction(transaction_id);

COMMENT ON COLUMN sat_transaction_party.transaction_id IS 'Transaction Identity';
COMMENT ON COLUMN sat_transaction_party.transaction_party_id IS 'Transaction Party Identity';


-- -----------------------------------------------------------
-- create view_transactions view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_transactions AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  s.beneficiary_name,
	s.original_account_name  AS orignator_name,
 	s.amount,
	s.debit_or_credit_code,
	s.currency_code AS beneficary_currency_code,
	s.original_currency_code AS origator_currency_code,
  h.transaction_id,
  h.orig_account_entity_id,
  h.bene_account_entity_id,
  h.transaction_entity_id,
  h.bene_account_id,
  h.orig_account_id,
  h.transaction_id_raw,
  h.period_start AS hub_period_start,
  h.period_end AS hub_period_end,
	s.type_code,
	s.type_concept_id,
	s.type_raw_code,
	s.type_raw_concept_id,
	s.debit_or_credit_concept_id,
	s.debit_or_credit_raw_code,
	s.debit_or_credit_raw_concept_id,
	s.counterparty_account_id,
	s.counterparty_account_transaction_id,
	s.is_self_transaction,
	s.is_international_transaction,
	s.datetime,
	s.description,
	s.balance_after,
	s.balance_before,	
	s.payment_method_code,
	s.payment_method_concept_id,
	s.payment_method_raw_code,
	s.payment_method_raw_concept_id,
	s.reference,
	s.account_sort_code,
	s.account_number,
	s.account_number_suffix,
 	s.iban,
	s.swiftbic,
	s.currency_concept_id,
	s.currency_raw_code,
	s.currency_raw_concept_id,
	s.country_code,
	s.country_concept_id,
	s.country_raw_code,
	s.country_raw_concept_id,
	s.original_amount,
	s.original_account_number,
	s.original_account_sort_code,
	s.original_iban,
	s.original_currency_concept_id,
	s.original_currency_raw_code,
	s.original_currency_raw_concept_id,
	s.original_country_code,
	s.original_country_concept_id,
	s.original_country_raw_code,
	s.original_country_raw_concept_id,
	s.original_bank,
	s.original_bank_country_code,
	s.institution_bank,
	s.institution_bank_country_code,
	s.sending_bank,
	s.sending_bank_country_code,
	s.beneficiary_bank,
	s.beneficiary_bank_country_code,
	s.payment_booking_date,
	s.payment_booking_system_code,
	s.payment_booking_system_concept_id,
	s.payment_booking_system_raw_code,
	s.payment_booking_system_raw_concept_id,
	s.payment_type_code,
	s.payment_type_concept_id,
	s.payment_type_raw_code,
	s.payment_type_raw_concept_id,
	s.other_receiver_information,
	s.payment_party_id,
	s.payment_account_number,
	s.payment_source_code,	
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

  FROM qdp_database.qdp_transactions_all.hub_transaction h
    JOIN qdp_database.qdp_transactions_all.sat_transaction s
      ON h.transaction_id = s.transaction_id
    JOIN qdp_database.qdp_refcodes_all.tennant t
      ON h.tennant_id = t.tennant_id
		ORDER BY s.beneficiary_name
;

SELECT * from view_transactions;

-- -----------------------------------------------------------
-- create view_account_beneficiary_transactions view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_account_beneficiary_transactions AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  h.account_id,
	h.account_entity_id,
--  icust.customer_name,
--  ihn.given1 AS individual_given_name,
--  ihn.family AS individual_family_name,
	h.period_start AS bank_account_opened,
	h.period_end AS bank_account_closed,
  s.sort_code,
  s.account_number,
  s.account_name,
  s.iban,
  s.swiftbic,
  addr.full_address AS bank_account_address,
  s.data_source_code,
  s.data_source_concept_id,
  s.data_source_raw_code,
  s.data_source_raw_concept_id,
  s.type_code,
  s.type_concept_id,
  s.type_raw_code,
  s.type_raw_concept_id,
  s.balance,
  s.overdraft_limit,
  s.loan_original_amount,
  s.loan_orininal_maturity_date,
  s.servicer_identification,
  s.servicer_scheme_name,    
  s.currency_code,
  s.currency_concept_id,
  s.currency_raw_code,
  s.currency_raw_concept_id,
  s.branch_code,
  s.branch_concept_id,
  s.branch_raw_code,
  s.branch_raw_concept_id,
  s.opened_datetime,
  s.closed_datetime,
  s.status_code,
  s.status_concept_id,
  s.status_raw_code,
  s.status_raw_concept_id,
  s.risk_rating,
  s.risk_rating_code,
  s.risk_rating_concept_id,
  s.risk_rating_raw_code,
  s.risk_rating_raw_concept_id,
  s.country_code,
  s.country_concept_id,
  s.country_raw_code,
  s.country_raw_concept_id,
  s.is_account_alarm,
  s.is_bank_account,
  s.is_business_account,
  s.is_customer_acccount,
  s.is_counterparty_account,
  s.is_frozed,
  s.is_closed,
  s.is_open,
  s.is_excluded_from_monitoring,
  s.sat_account_id,
	s.load_datetime,
	s.record_source_id,
  s.individual_customer_entity_id,  	
  s.individual_customer_id,
  s.organisation_customer_entity_id,  	
  s.organisation_customer_id,
  s.address_entity_id,
  s.address_id,
  s.product_entity_id,
  s.product_id,
	s.period_start,
	s.period_end,
  htrans.transaction_id,
  htrans.transaction_entity_id,
  htrans.transaction_id_raw,
  htrans.period_start AS hub_trans_period_start,
  htrans.period_end AS hub_trans_period_end,
	strans.type_code AS trans_type_code,
	strans.type_concept_id AS trans_type_concept_id,
	strans.type_raw_code AS trans_type_raw_code,
	strans.type_raw_concept_id AS trans_type_raw_concept_id,
	strans.debit_or_credit_code,
	strans.debit_or_credit_concept_id,
	strans.debit_or_credit_raw_code,
	strans.debit_or_credit_raw_concept_id,
	strans.counterparty_account_id,
	strans.counterparty_account_transaction_id,
	strans.is_self_transaction,
	strans.is_international_transaction,
 	strans.amount,
	strans.datetime,
	strans.description,
	strans.balance_after,
	strans.balance_before,	
	strans.payment_method_code,
	strans.payment_method_concept_id,
	strans.payment_method_raw_code,
	strans.payment_method_raw_concept_id,
	strans.reference,
	strans.account_sort_code,
	strans.account_number AS trans_account_number,
	strans.account_number_suffix,
 	strans.iban AS trans_iban,
	strans.swiftbic AS trans_swiftbic,
	strans.beneficiary_name,
	strans.currency_code AS trans_currency_code,
	strans.currency_concept_id AS trans_currency_concept_id,
	strans.currency_raw_code AS trans_currency_raw_code,
	strans.currency_raw_concept_id AS trans_currency_raw_concept_id,
	strans.country_code AS trans_country_code,
	strans.country_concept_id AS trans_country_concept_id,
	strans.country_raw_code AS trans_country_raw_code,
	strans.country_raw_concept_id AS  trans_country_raw_concept_id,
	strans.original_amount,
	strans.original_account_number,
	strans.original_account_sort_code,
	strans.original_iban,
	strans.original_account_name,
	strans.original_currency_code,
	strans.original_currency_concept_id,
	strans.original_currency_raw_code,
	strans.original_currency_raw_concept_id,
	strans.original_country_code,
	strans.original_country_concept_id,
	strans.original_country_raw_code,
	strans.original_country_raw_concept_id,
	strans.original_bank,
	strans.original_bank_country_code,
	strans.institution_bank,
	strans.institution_bank_country_code,
	strans.sending_bank,
	strans.sending_bank_country_code,
	strans.beneficiary_bank,
	strans.beneficiary_bank_country_code,
	strans.payment_booking_date,
	strans.payment_booking_system_code,
	strans.payment_booking_system_concept_id,
	strans.payment_booking_system_raw_code,
	strans.payment_booking_system_raw_concept_id,
	strans.payment_type_code,
	strans.payment_type_concept_id,
	strans.payment_type_raw_code,
	strans.payment_type_raw_concept_id,
	strans.other_receiver_information,
	strans.payment_party_id,
	strans.payment_account_number,
	strans.payment_source_code,	
	strans.period_start AS sat_trans_period_start,
	strans.period_end AS sat_trans_period_end,
	strans.load_datetime AS trans_load_datetime,
	strans.record_source_id AS trans_record_source_id,
	strans.data_source_code AS trans_data_source_code,
	strans.data_source_concept_id AS trans_data_source_concept_id,
	strans.data_source_raw_code AS trans_data_source_raw_code,
	strans.data_source_raw_concept_id AS trans_data_source_raw_concept_id


FROM qdp_database.qdp_accounts_all.hub_account h
JOIN qdp_database.qdp_accounts_all.sat_account s
  ON h.account_id = s.account_id
--JOIN qdp_database.qdp_individual_customers.sat_individual_customers icust
--  ON s.individual_customer_id = icust.individual_customer_id
--JOIN qdp_database.qdp_individuals_all.sat_human_name ihn
--  ON icust.individual_id = ihn.individual_id
JOIN qdp_database.qdp_locations_all.sat_address addr
  ON s.address_id = addr.address_id
JOIN qdp_database.qdp_transactions_all.hub_transaction htrans
  ON htrans.bene_account_id = h.account_id
JOIN qdp_database.qdp_transactions_all.sat_transaction strans
  ON htrans.transaction_id = strans.transaction_id
JOIN qdp_database.qdp_refcodes_all.tennant t
  ON h.tennant_id = t.tennant_id

-- WHERE ihn.is_trusted = true
ORDER BY s.account_id ASC
;


SELECT * from view_account_beneficiary_transactions;


-- -----------------------------------------------------------
-- create view_account_originator_transactions view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_account_originator_transactions AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  h.account_id,
	h.account_entity_id,
--  icust.customer_name,
--  ihn.given1 AS individual_given_name,
--  ihn.family AS individual_family_name,
	h.period_start AS bank_account_opened,
	h.period_end AS bank_account_closed,
  s.sort_code,
  s.account_number,
  s.account_name,
  s.iban,
  s.swiftbic,
  addr.full_address AS bank_account_address,
  s.data_source_code,
  s.data_source_concept_id,
  s.data_source_raw_code,
  s.data_source_raw_concept_id,
  s.type_code,
  s.type_concept_id,
  s.type_raw_code,
  s.type_raw_concept_id,
  s.balance,
  s.overdraft_limit,
  s.loan_original_amount,
  s.loan_orininal_maturity_date,
  s.servicer_identification,
  s.servicer_scheme_name,    
  s.currency_code,
  s.currency_concept_id,
  s.currency_raw_code,
  s.currency_raw_concept_id,
  s.branch_code,
  s.branch_concept_id,
  s.branch_raw_code,
  s.branch_raw_concept_id,
  s.opened_datetime,
  s.closed_datetime,
  s.status_code,
  s.status_concept_id,
  s.status_raw_code,
  s.status_raw_concept_id,
  s.risk_rating,
  s.risk_rating_code,
  s.risk_rating_concept_id,
  s.risk_rating_raw_code,
  s.risk_rating_raw_concept_id,
  s.country_code,
  s.country_concept_id,
  s.country_raw_code,
  s.country_raw_concept_id,
  s.is_account_alarm,
  s.is_bank_account,
  s.is_business_account,
  s.is_customer_acccount,
  s.is_counterparty_account,
  s.is_frozed,
  s.is_closed,
  s.is_open,
  s.is_excluded_from_monitoring,
  s.sat_account_id,
	s.load_datetime,
	s.record_source_id,
  s.individual_customer_entity_id,  	
  s.individual_customer_id,
  s.organisation_customer_entity_id,  	
  s.organisation_customer_id,
  s.address_entity_id,
  s.address_id,
  s.product_entity_id,
  s.product_id,
	s.period_start,
	s.period_end,
  htrans.transaction_id,
  htrans.transaction_entity_id,
  htrans.transaction_id_raw,
  htrans.period_start AS hub_trans_period_start,
  htrans.period_end AS hub_trans_period_end,
	strans.type_code AS trans_type_code,
	strans.type_concept_id AS trans_type_concept_id,
	strans.type_raw_code AS trans_type_raw_code,
	strans.type_raw_concept_id AS trans_type_raw_concept_id,
	strans.debit_or_credit_code,
	strans.debit_or_credit_concept_id,
	strans.debit_or_credit_raw_code,
	strans.debit_or_credit_raw_concept_id,
	strans.counterparty_account_id,
	strans.counterparty_account_transaction_id,
	strans.is_self_transaction,
	strans.is_international_transaction,
 	strans.amount,
	strans.datetime,
	strans.description,
	strans.balance_after,
	strans.balance_before,	
	strans.payment_method_code,
	strans.payment_method_concept_id,
	strans.payment_method_raw_code,
	strans.payment_method_raw_concept_id,
	strans.reference,
	strans.account_sort_code,
	strans.account_number AS trans_account_number,
	strans.account_number_suffix,
 	strans.iban AS trans_iban,
	strans.swiftbic AS trans_swiftbic,
	strans.beneficiary_name,
	strans.currency_code AS trans_currency_code,
	strans.currency_concept_id AS trans_currency_concept_id,
	strans.currency_raw_code AS trans_currency_raw_code,
	strans.currency_raw_concept_id AS trans_currency_raw_concept_id,
	strans.country_code AS trans_country_code,
	strans.country_concept_id AS trans_country_concept_id,
	strans.country_raw_code AS trans_country_raw_code,
	strans.country_raw_concept_id AS  trans_country_raw_concept_id,
	strans.original_amount,
	strans.original_account_number,
	strans.original_account_sort_code,
	strans.original_iban,
	strans.original_account_name,
	strans.original_currency_code,
	strans.original_currency_concept_id,
	strans.original_currency_raw_code,
	strans.original_currency_raw_concept_id,
	strans.original_country_code,
	strans.original_country_concept_id,
	strans.original_country_raw_code,
	strans.original_country_raw_concept_id,
	strans.original_bank,
	strans.original_bank_country_code,
	strans.institution_bank,
	strans.institution_bank_country_code,
	strans.sending_bank,
	strans.sending_bank_country_code,
	strans.beneficiary_bank,
	strans.beneficiary_bank_country_code,
	strans.payment_booking_date,
	strans.payment_booking_system_code,
	strans.payment_booking_system_concept_id,
	strans.payment_booking_system_raw_code,
	strans.payment_booking_system_raw_concept_id,
	strans.payment_type_code,
	strans.payment_type_concept_id,
	strans.payment_type_raw_code,
	strans.payment_type_raw_concept_id,
	strans.other_receiver_information,
	strans.payment_party_id,
	strans.payment_account_number,
	strans.payment_source_code,	
	strans.period_start AS sat_trans_period_start,
	strans.period_end AS sat_trans_period_end,
	strans.load_datetime AS trans_load_datetime,
	strans.record_source_id AS trans_record_source_id,
	strans.data_source_code AS trans_data_source_code,
	strans.data_source_concept_id AS trans_data_source_concept_id,
	strans.data_source_raw_code AS trans_data_source_raw_code,
	strans.data_source_raw_concept_id AS trans_data_source_raw_concept_id


FROM qdp_database.qdp_accounts_all.hub_account h
JOIN qdp_database.qdp_accounts_all.sat_account s
  ON h.account_id = s.account_id
--JOIN qdp_database.qdp_individual_customers.sat_individual_customers icust
--  ON s.individual_customer_id = icust.individual_customer_id
--JOIN qdp_database.qdp_individuals_all.sat_human_name ihn
--  ON icust.individual_id = ihn.individual_id
JOIN qdp_database.qdp_locations_all.sat_address addr
  ON s.address_id = addr.address_id
JOIN qdp_database.qdp_transactions_all.hub_transaction htrans
  ON htrans.orig_account_id = h.account_id
JOIN qdp_database.qdp_transactions_all.sat_transaction strans
  ON htrans.transaction_id = strans.transaction_id
JOIN qdp_database.qdp_refcodes_all.tennant t
  ON h.tennant_id = t.tennant_id

--WHERE ihn.is_trusted = true
ORDER BY s.account_id ASC
;


SELECT * from view_account_originator_transactions;


-- -----------------------------------------------------------
-- create view_account_transactions view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_account_transactions AS 
SELECT 'Beneficiary' AS BeneficaryOrOriginator, * from view_account_beneficiary_transactions
UNION ALL
SELECT 'Originator' AS BeneficaryOrOriginator, * from view_account_originator_transactions
;

SELECT * FROM view_account_transactions;


